In [1]:
import os
import numpy as np
import cv2
from scipy.ndimage import gaussian_filter

## User defined parameters

In [2]:
gt_path = "Your path to the ground truth images"
input_path = "Your path to the input images"
NUM = your number of images
output_size = (256,256)

## Code

In [3]:
os.makedirs(gt_path, exist_ok=True)
os.makedirs(input_path, exist_ok=True)

#### Generate the test data


In [4]:
# bernstein_poly 
def bernstein_poly(t, n, i):
    return np.math.comb(n, i) * (t**i) * ((1 - t)**(n - i))

def bezier_curve(control_points, num_points=100):
    n = len(control_points) - 1  
    t_values = np.linspace(0, 1, num_points)
    curve = np.zeros((num_points, 2))  

    for i in range(n + 1):
        curve += np.array([bernstein_poly(t, n, i) for t in t_values]).reshape(-1, 1) * control_points[i]

    return curve

# bezier_curve
def generate_random_bezier_curve(image_size):
    control_points = np.random.randint(10, image_size - 10, size=(4, 2))  
    curve = bezier_curve(control_points, num_points=200)

    curve_image = np.zeros((image_size, image_size), dtype=np.uint8)
    for i in range(len(curve) - 1):
        pt1 = tuple(curve[i].astype(int))
        pt2 = tuple(curve[i + 1].astype(int))
        cv2.line(curve_image, pt1, pt2, 255, 1)  
    return curve_image, control_points

# generate random lines
def generate_curves_and_lines(image_size):
    img = np.zeros((image_size, image_size), dtype=np.uint8)
    gt_data = []
    used_columns = set()
    
    # generate random curves
    for _ in range(np.random.randint(5, 21)):  
        curve_img, control_points = generate_random_bezier_curve(image_size)
        img = cv2.add(img, curve_img)
        gt_data.append(("curve", control_points))  

    # generate array lines
    num_lines = np.random.randint(10, 61)
    for _ in range(num_lines):
        max_attempts = 100  # maximum number of attempts to place a line
        attempts = 0
        while attempts < max_attempts:
            attempts += 1
            x1 = np.random.randint(0, width)  # generate a random column
            # check if the column is not too close to the previous lines
            if all(abs(x1 - used_col) >= 3 for used_col in used_columns):
                y1 = np.random.randint(0, 50)  # generate a random row
                length = np.random.randint(200, 401)  # generate a random length
                x2, y2 = x1, min(y1 + length, height - 1)  # calculate the end of the line
                gt_data.append(((x1, y1), (x2, y2)))
                cv2.line(img, (x1, y1), (x2, y2), 255, 1)
                used_columns.add(x1)  # remember the column
                break
        else:
            print(f"Warning: Could not place a line after {max_attempts} attempts.")
            
    return img, gt_data

#### Apply brightness and blur


In [5]:

def apply_brightness_and_blur(img, gt_data):
    blurred_img = np.zeros_like(img, dtype=np.float32)
    for item in gt_data:
        if item[0] == "line":  
            ((x1, y1), (x2, y2)) = item[1]
            brightness = np.random.randint(150, 256)
            temp_img = np.zeros_like(img, dtype=np.uint8)
            cv2.line(temp_img, (x1, y1), (x2, y2), brightness, 1)
            sigma = np.random.uniform(0.8, 2.4)
            temp_img = gaussian_filter(temp_img.astype(np.float32), sigma=sigma)
            blurred_img += temp_img
        elif item[0] == "curve":  
            control_points = item[1]
            brightness = np.random.randint(150, 256)
            curve = bezier_curve(control_points, num_points=200)
            temp_img = np.zeros_like(img, dtype=np.uint8)
            for i in range(len(curve) - 1):
                pt1 = tuple(curve[i].astype(int))
                pt2 = tuple(curve[i + 1].astype(int))
                cv2.line(temp_img, pt1, pt2, brightness, 1)
            sigma = np.random.uniform(0.8, 3.6)
            temp_img = gaussian_filter(temp_img.astype(np.float32), sigma=sigma)
            blurred_img += temp_img
    return np.clip(blurred_img, 0, 255).astype(np.uint8)

#### Add background and noise


In [6]:

def add_background_and_noise(img):
    background_value = np.random.randint(30, 121)
    background = np.full_like(img, background_value, dtype=np.uint8)
    noisy_img = cv2.addWeighted(img, 1, background, 1, 0)

    
    noise = np.random.normal(0, 10, img.shape).astype(np.float32)
    noisy_img = noisy_img.astype(np.float32) + noise
    return np.clip(noisy_img, 0, 255).astype(np.uint8)

#### Main

In [ ]:

for i in range(NUM):  # generate X images
    base_img, gt_data = generate_curves_and_lines(image_size)

    # Save ground truth
    gt_save_path = os.path.join(gt_path, f"randomgt_{i:04d}.png")
    cv2.imwrite(gt_save_path, base_img)

    # add light stripes background
    blurred_img = apply_brightness_and_blur(base_img, gt_data)

    # add noise
    noisy_img = add_background_and_noise(blurred_img)

    # save input image
    input_save_path = os.path.join(input_path, f"randominput_{i:04d}.png")
    cv2.imwrite(input_save_path, noisy_img)